# 🛠️ Practice Project Template 05: Bidirectional LSTM Text Classifier
## Hands-On Implementation: Deep Sequence Modeling in PyTorch

Welcome to your hands-on project template! This notebook is designed for **active practice**. It provides the complete architectural framework, problem description, data loading, guided `TODO` code blocks, and an expandable solution key so you can build, train, and test a deep bidirectional sequence classifier yourself.

---

### 📋 Project Objective:
Build an end-to-end sentiment classification model using a **Bidirectional LSTM** in PyTorch.
1. Build a custom vocabulary and integer indexer with `<PAD>` and `<UNK>` tokens.
2. Implement a custom PyTorch `Dataset` with padding and truncation.
3. Design a `BiLSTMClassifier` neural network with:
   - `nn.Embedding`
   - Multi-layer `nn.LSTM(bidirectional=True)`
   - Dropout and Linear classification head
4. Train the model with CrossEntropyLoss and Adam optimizer.
5. Create an interactive inference function that predicts the sentiment and confidence score of any custom user sentence!


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


---
## Step 1: The Dataset
Here is our domain dataset consisting of technical product and course feedback:


In [ ]:
raw_dataset = [
    ("The explanations are crystal clear and the math is very intuitive.", 1),
    ("Loved the deep learning projects and clear visual diagrams.", 1),
    ("Outstanding tutorial, the code runs perfectly without any issues.", 1),
    ("Super helpful walkthrough, my understanding improved drastically.", 1),
    ("Brilliant content, well structured and very practical.", 1),
    ("Highly recommend this to anyone learning machine learning from scratch.", 1),
    ("The exercises are engaging and the explanations make sense.", 1),
    ("The code is completely broken and full of syntax errors.", 0),
    ("Terrible quality, outdated libraries and very confusing explanations.", 0),
    ("Waste of time, does not explain the core concepts at all.", 0),
    ("Very disappointing lesson, frustrating and unhelpful.", 0),
    ("Horrible pacing, skipped all the important mathematical derivations.", 0),
    ("Could not get any of the examples to run, awful documentation.", 0),
    ("Completely unreadable code with poor variable names and bugs.", 0)
]

print(f"Total labeled reviews: {len(raw_dataset)}")


---
## Step 2: [YOUR TURN] Build Vocabulary & Custom Tokenizer
**Instructions:**
- Write a `tokenize(text)` helper function that converts text to lowercase and extracts words.
- Build a vocabulary dictionary `vocab` mapping `<PAD>` to `0`, `<UNK>` to `1`, and each unique word to a sequential index.


In [ ]:
# TODO: Implement tokenize and vocab dictionary
def tokenize(text):
    ### YOUR CODE HERE ###
    return re.findall(r'\b\w+\b', text.lower())

vocab = {"<PAD>": 0, "<UNK>": 1}

### YOUR CODE HERE: Iterate over raw_dataset and populate vocab ###
for text, _ in raw_dataset:
    for word in tokenize(text):
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)} tokens")
print(f"Sample tokens: {list(vocab.items())[:10]}")


---
## Step 3: [YOUR TURN] Create Custom PyTorch `Dataset`
**Instructions:**
- Subclass `torch.utils.data.Dataset`.
- In `__getitem__`, convert words to token IDs using `vocab.get(word, vocab['<UNK>'])`.
- Pad or truncate the sequence to `max_len` using `vocab['<PAD>']`.


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, data, vocab, max_len=16):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        text, label = self.data[idx]
        tokens = tokenize(text)
        
        ### YOUR CODE HERE: Convert tokens to IDs and pad/truncate to max_len ###
        token_ids = [self.vocab.get(tok, self.vocab["<UNK>"]) for tok in tokens]
        if len(token_ids) < self.max_len:
            token_ids = token_ids + [self.vocab["<PAD>"]] * (self.max_len - len(token_ids))
        else:
            token_ids = token_ids[:self.max_len]
            
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# Verification
dataset = SentimentDataset(raw_dataset, vocab, max_len=14)
loader = DataLoader(dataset, batch_size=4, shuffle=True)
sample_x, sample_y = next(iter(loader))
print(f"Sample batch X shape: {sample_x.shape}")
print(f"Sample batch Y shape: {sample_y.shape}")


---
## Step 4: [YOUR TURN] Define the Bidirectional LSTM Architecture
**Instructions:**
- In `__init__`, define `nn.Embedding`, `nn.LSTM(bidirectional=True, batch_first=True)`, and a classification head `nn.Sequential`.
- In `forward(x)`, embed the input, pass through the LSTM, extract the final forward and backward hidden states, concatenate them, and pass to the classifier.


In [ ]:
class BiLSTMSentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=32, num_classes=2, dropout=0.2):
        super().__init__()
        
        ### YOUR CODE HERE: Define layers ###
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        ### YOUR CODE HERE: Forward pass ###
        embedded = self.embedding(x)
        output, (h_n, c_n) = self.lstm(embedded)
        
        # h_n shape: (num_layers * num_directions, batch_size, hidden_dim)
        forward_final = h_n[-2, :, :]
        backward_final = h_n[-1, :, :]
        combined = torch.cat((forward_final, backward_final), dim=1)
        
        logits = self.classifier(combined)
        return logits

model = BiLSTMSentimentModel(vocab_size=len(vocab), embed_dim=32, hidden_dim=32).to(device)
print(model)


---
## Step 5: Train the Model & Run Live Inference
Let's run the training loop and test real-time predictions!


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
model.train()
for epoch in range(1, 31):
    total_loss = 0.0
    correct = 0
    total = 0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model(bx)
        loss = criterion(out, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        total_loss += loss.item() * bx.size(0)
        preds = torch.argmax(out, dim=1)
        correct += (preds == by).sum().item()
        total += by.size(0)
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/30 | Loss: {total_loss/total:.4f} | Accuracy: {correct/total * 100:.1f}%")


In [ ]:
# Live Prediction Function
def predict(text):
    model.eval()
    tokens = tokenize(text)
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    if len(ids) < 14:
        ids += [vocab["<PAD>"]] * (14 - len(ids))
    else:
        ids = ids[:14]
        
    tensor_in = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(tensor_in)
        prob = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = np.argmax(prob)
        
    label_str = "POSITIVE 🟢" if pred == 1 else "NEGATIVE 🔴"
    print(f"Input: '{text}'")
    print(f"Result: {label_str} (Confidence: {prob[pred]*100:.2f}%)\n")

predict("This is an exceptional tutorial with brilliant explanations.")
predict("Broken examples and horrible bugs, complete waste.")


---
## 🏆 Practice Challenge Extensions:
1. **Add Attention Pooling**: Instead of taking the final hidden state, compute an attention-weighted sum of all time steps!
2. **Experiment with GRU**: Replace `nn.LSTM` with `nn.GRU` and compare the parameter count and training speed.
